In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
import os
import models.unet_precip_regression_lightning as unet_regr
from models import SmaAT_UNet_VQ_lightning
from utils import dataset_precip
from argparse import Namespace

dataset = dataset_precip.precipitation_maps_oversampled_h5(
            in_file="data/precipitation/train_test_2016-2019_input-length_12_img-ahead_6_rain-threshold_50.h5",
            num_input_images=12,
            num_output_images=6, train=False)

index_seq = range(1, len(dataset), 5)   # every 5-th sample
models_dir = "lightning/precip_regression/comparison"
precip_models = [m for m in os.listdir(models_dir) if ".ckpt" in m]
print("Found checkpoints:", precip_models)

loss_func = nn.MSELoss(reduction="sum")
scaling  = 47.83                         
TOP_K    = 30                            

def load_model_with_type(model_cls, model_file, model_type=None):
    """Load a checkpoint to *CPU* with optional 'model_type' override."""
    ckpt_path = f"{models_dir}/{model_file}"
    checkpoint = torch.load(ckpt_path, map_location="cpu")

    hparams_dict = checkpoint.get("hyper_parameters", {})
    if model_type is not None:
        hparams_dict["model_type"] = model_type
    hparams_ns = Namespace(**hparams_dict)

    model = model_cls(hparams_ns)
    state_dict = checkpoint.get("state_dict", checkpoint)
    state_dict = {k[len("model."):] if k.startswith("model.") else k: v
                  for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    return model

diff_list = [] 

for idx in index_seq:
    print(f"Processing sample {idx} …")
    x, y_true = dataset[idx]
    x      = torch.tensor(x).unsqueeze(0)          # (1,C,H,W)
    y_true = torch.tensor(y_true).unsqueeze(0)     # (1,6,H,W) but we only need first pred map
    y_true_map = y_true.squeeze()                  # (6,H,W) use whole stack for MSE

    smiq_loss, smaat_loss = None, None

    for model_file in precip_models:
        if   "SmaAT_UNet_VQ_MSE-MixConv" in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SMiQ-UNet", SmaAT_UNet_VQ_lightning.SmaAT_UNet_VQ, "partialmixconv", True
        elif "SmaAT_UNet_VQ_MSE" in model_file and "MixConv" not in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SQ-UNet", SmaAT_UNet_VQ_lightning.SmaAT_UNet_VQ, "nomixconv", True
        elif "SmaAT_UNet_MixConv" in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SMi-UNet", SmaAT_UNet_VQ_lightning.SmaAT_UNet_MixConv, None, False
        elif "SmaAT_UNet" in model_file and "VQ" not in model_file and "MixConv" not in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SmaAT-UNet", unet_regr.UNetDS_Attention, None, False
        else:
            continue

        if model_type is None:
            model = model_cls.load_from_checkpoint(
                        f"{models_dir}/{model_file}", map_location="cpu")
        else:
            model = load_model_with_type(model_cls, model_file, model_type)
        model.to("cpu").eval()

        # forward pass 
        with torch.no_grad():
            y_pred = model(x)[0] if use_vq else model(x)   # (1,6,H,W)
        mse_val = loss_func(y_pred.squeeze()*scaling,
                            y_true_map*scaling).item()

        if   model_name == "SMiQ-UNet":  smiq_loss  = mse_val
        elif model_name == "SmaAT-UNet": smaat_loss = mse_val

    if smiq_loss is not None and smaat_loss is not None:
        diff = smaat_loss - smiq_loss
        diff_list.append((idx, diff, smiq_loss, smaat_loss))

diff_list.sort(key=lambda t: t[1], reverse=True)
print(f"\nTop-{TOP_K} samples where (SmaAT − SMiQ) MSE is largest:")
for r, (i,d,sq,sa) in enumerate(diff_list[:TOP_K], 1):
    print(f"{r:2d}) sample {i:4d} | ΔMSE={d:.4f}  (SMiQ={sq:.4f}, SmaAT={sa:.4f})")

top_indices = [t[0] for t in diff_list[:TOP_K]]

# plot top-k
order = ["SMiQ-UNet", "SMi-UNet", "SQ-UNet", "SmaAT-UNet"]

for idx in top_indices:
    x, y_true = dataset[idx]
    x      = torch.tensor(x).unsqueeze(0)
    y_true = torch.tensor(y_true).unsqueeze(0)
    y_phys = y_true.squeeze()*scaling
    vmin, vmax = y_phys.min().item(), y_phys.max().item()

    plot_images = {}

    # run all four models (same logic as above, but we need their outputs for plotting)
    for model_file in precip_models:
        if   "SmaAT_UNet_VQ_MSE-MixConv" in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SMiQ-UNet", SmaAT_UNet_VQ_lightning.SmaAT_UNet_VQ, "partialmixconv", True
        elif "SmaAT_UNet_VQ_MSE" in model_file and "MixConv" not in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SQ-UNet", SmaAT_UNet_VQ_lightning.SmaAT_UNet_VQ, "nomixconv", True
        elif "SmaAT_UNet_MixConv" in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SMi-UNet", SmaAT_UNet_VQ_lightning.SmaAT_UNet_MixConv, None, False
        elif "SmaAT_UNet" in model_file and "VQ" not in model_file and "MixConv" not in model_file:
            model_name, model_cls, model_type, use_vq = \
                "SmaAT-UNet", unet_regr.UNetDS_Attention, None, False
        else:
            continue

        if model_type is None:
            model = model_cls.load_from_checkpoint(
                        f"{models_dir}/{model_file}", map_location="cpu")
        else:
            model = load_model_with_type(model_cls, model_file, model_type)
        model.to("cpu").eval()

        with torch.no_grad():
            y_pred = model(x)[0] if use_vq else model(x)
        plot_images[model_name] = y_pred.squeeze()

    # ---- figure ---- 
    fig, axes = plt.subplots(1, 1+len(order), figsize=(20,4))
    axes[0].imshow(y_phys, vmin=vmin, vmax=vmax, origin='upper')
    axes[0].set_title(f"GT (sample {idx})", fontsize=12)
    axes[0].set_xticks([0,100,200]); axes[0].set_yticks([0,100,200])

    for i, mname in enumerate(order, 1):
        axes[i].imshow(plot_images[mname].cpu().numpy()*scaling,
                       vmin=vmin, vmax=vmax, origin='upper')
        axes[i].set_title(mname, fontsize=12)
        axes[i].set_xticks([0,100,200]); axes[i].set_yticks([0,100,200])

    plt.tight_layout()
    plt.show()
